# MIT.pdf Retrieval Lab

이 노트북은 `weeks/week02/data/MIT.pdf`를 가지고 2주차 개념을 직접 실습해 보는 자료입니다.

## 이번 실습에서 할 것

- 실제 임베딩 모델로 `MIT.pdf` 문서를 벡터화해 보기
- 간단한 in-memory Vector Store에 저장해 similarity search 실행해 보기
- chunking 방식에 따라 retrieval 결과가 어떻게 달라지는지 비교해 보기
- dense only, sparse only, hybrid search 결과 차이를 체감해 보기

## 실습 포인트

- `MIT.pdf`는 기억(memory)과 신경과학(neuroscience) 관련 글입니다.
- dense search는 의미 기반 검색에 강하고,
- sparse search는 정확한 단어 매칭에 강하며,
- hybrid search는 둘의 장점을 합치려는 접근입니다.


## 권장 학습 순서

1. 문서를 불러오고 텍스트를 확인합니다.
2. chunking 전략을 2~3개로 나눠 봅니다.
3. dense embedding + vector store를 만듭니다.
4. sparse search를 추가합니다.
5. hybrid search를 만들고 결과를 비교합니다.
6. 마지막으로 chunking 전략별 hit@k 차이를 봅니다.

이 노트북은 **직접 바꿔 가면서 실험하는 것**이 가장 중요합니다.


In [1]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "pypdf": "pypdf",
    "sentence_transformers": "sentence-transformers",
}

missing = [pip_name for module_name, pip_name in REQUIRED.items() if importlib.util.find_spec(module_name) is None]

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")


Installing: ['sentence-transformers']



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path
from textwrap import shorten

import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter


In [3]:
CANDIDATE_PATHS = [
    Path("weeks/week02/data/MIT.pdf"),
    Path("../data/MIT.pdf"),
    Path("data/MIT.pdf"),
]

PDF_PATH = None
for candidate in CANDIDATE_PATHS:
    if candidate.exists():
        PDF_PATH = candidate.resolve()
        break

if PDF_PATH is None:
    raise FileNotFoundError("MIT.pdf를 찾지 못했습니다. week02/data/MIT.pdf 경로를 확인하세요.")

PDF_PATH


PosixPath('/Users/hong-yuseog/Desktop/RAGRAG/weeks/week02/data/MIT.pdf')

## 1. PDF 읽기

먼저 `MIT.pdf`를 페이지 단위로 읽습니다. 여기서는 각 페이지를 하나의 원본 문서 단위로 보고, 이후 chunking 단계에서 더 잘게 나눌 것입니다.


In [4]:
reader = PdfReader(str(PDF_PATH))

pages = []
for page_number, page in enumerate(reader.pages, start=1):
    raw_text = page.extract_text() or ""
    cleaned_text = " ".join(raw_text.split())
    pages.append({"page": page_number, "text": cleaned_text})

print(f"pages: {len(pages)}")
print(f"total characters: {sum(len(page['text']) for page in pages):,}")


pages: 6
total characters: 12,168


In [5]:
for page in pages[:2]:
    print(f"\n--- page {page['page']} preview ---")
    print(shorten(page['text'], width=900, placeholder=" ..."))



--- page 1 preview ---
1 MIT Vol.2-2 How technology can let us see and manipulate memories By Joshua Sariñana Optogenetics and advanced imaging have helped neuroscientists understand how memories form and made it possible to manipulate them. In Your Own Words There are 86 billion neurons in the human brain, each with thousands of connections, giving rise to hundreds of trillions of synapses. Synapses—the connection points between neurons—store memories. The overwhelming number of neurons and synapses in our brains makes finding the precise location of a specific memory a formidable scientific challenge. Figuring out how memories form may ultimately help us learn more about ourselves and keep our mental acuity intact. Memory helps shape our identities, and memory impairment may indicate a brain disorder. Alzheimer’s disease robs individuals of their memories by destroying synapses; addiction hijacks the ...

--- page 2 preview ---
2 Episodic autobiographical memory deals with what happ

위 미리보기에서 볼 수 있듯, 문서는 기억(memory), 기억 흔적(memory engram), 도구(optogenetics, imaging), 거짓 기억(false memory) 같은 주제를 다룹니다.

이제 이 문서를 여러 chunking 방식으로 나눠 보고, 검색 결과가 어떻게 달라지는지 비교합니다.


## 2. Chunking 전략 준비

이번 실습에서는 아래 세 가지 전략을 씁니다.

- `recursive_small`: 작은 청크, overlap 있음
- `recursive_large`: 큰 청크, overlap 있음
- `token_medium`: 토큰 기준 청크

핵심 비교 포인트는 다음과 같습니다.

- 청크가 작을수록 검색 정밀도는 좋아질 수 있지만 문맥이 끊길 수 있음
- 청크가 클수록 문맥은 풍부하지만 관련 없는 정보가 섞일 수 있음
- token 기준 split은 모델 입력 길이 관리에 유리함


In [6]:
splitters = {
    "recursive_small": RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=80,
        separators=["\n\n", "\n", ". ", " ", ""],
    ),
    "recursive_large": RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=120,
        separators=["\n\n", "\n", ". ", " ", ""],
    ),
    "token_medium": TokenTextSplitter(
        chunk_size=180,
        chunk_overlap=40,
    ),
}

def build_chunks(pages, splitter_name):
    splitter = splitters[splitter_name]
    chunks = []
    for page in pages:
        split_texts = splitter.split_text(page["text"])
        for chunk_idx, chunk_text in enumerate(split_texts, start=1):
            chunks.append(
                {
                    "chunk_id": f"p{page['page']}-c{chunk_idx}",
                    "page": page["page"],
                    "strategy": splitter_name,
                    "text": chunk_text,
                    "char_len": len(chunk_text),
                }
            )
    return chunks

chunk_sets = {name: build_chunks(pages, name) for name in splitters}


In [7]:
for name, chunks in chunk_sets.items():
    avg_len = sum(chunk["char_len"] for chunk in chunks) / len(chunks)
    print(f"{name:16} chunks={len(chunks):2d} avg_char_len={avg_len:.1f}")


recursive_small  chunks=41 avg_char_len=305.7
recursive_large  chunks=19 avg_char_len=668.9
token_medium     chunks=21 avg_char_len=717.6


In [8]:
strategy_to_preview = "recursive_large"
for chunk in chunk_sets[strategy_to_preview][:3]:
    print(f"\n[{chunk['chunk_id']}] page={chunk['page']} len={chunk['char_len']}")
    print(shorten(chunk["text"], width=500, placeholder=" ..."))



[p1-c1] page=1 len=790
1 MIT Vol.2-2 How technology can let us see and manipulate memories By Joshua Sariñana Optogenetics and advanced imaging have helped neuroscientists understand how memories form and made it possible to manipulate them. In Your Own Words There are 86 billion neurons in the human brain, each with thousands of connections, giving rise to hundreds of trillions of synapses. Synapses—the connection points between neurons—store memories. The overwhelming number of neurons and synapses in our brains ...

[p1-c2] page=1 len=761
. Memory helps shape our identities, and memory impairment may indicate a brain disorder. Alzheimer’s disease robs individuals of their memories by destroying synapses; addiction hijacks the brain’s learning and memory centers; and some mental health conditions, like depression, are associated with memory impairment. Summary → The sheer number of neurons and synapses in the human brain makes it incredibly difficult for scientists to locate a speci

## 3. 실제 임베딩 모델로 벡터화하기

여기서는 `intfloat/multilingual-e5-small` 모델을 사용합니다.

이 모델을 고른 이유:

- 영어 문서 검색이 가능하고
- 한국어 질의도 어느 정도 다룰 수 있으며
- 문서(`passage:`)와 질의(`query:`) prefix를 붙여 retrieval 용도로 쓰기 좋기 때문입니다.

처음 실행 시 모델 다운로드 때문에 시간이 걸릴 수 있습니다.


In [9]:
EMBED_MODEL_NAME = "intfloat/multilingual-e5-small"
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

def embed_passages(texts, batch_size=32):
    prefixed = [f"passage: {text}" for text in texts]
    return embed_model.encode(
        prefixed,
        normalize_embeddings=True,
        batch_size=batch_size,
        show_progress_bar=True,
    )

def embed_queries(texts):
    prefixed = [f"query: {text}" for text in texts]
    return embed_model.encode(
        prefixed,
        normalize_embeddings=True,
        show_progress_bar=False,
    )


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

## 4. 간단한 Dense Vector Store 만들기

이번 실습에서는 복잡한 DB 대신, 학습용으로 **in-memory vector store**를 만듭니다.

핵심은 아래 두 가지입니다.

- chunk 텍스트 + 메타데이터를 보관하고
- 임베딩 벡터와 코사인 유사도로 top-k 검색을 수행합니다.

이렇게 해도 Vector Store의 기본 원리를 이해하기에는 충분합니다.


In [10]:
class SimpleDenseVectorStore:
    def __init__(self, chunks, embeddings):
        self.chunks = chunks
        self.embeddings = np.asarray(embeddings, dtype="float32")
        self.id_to_chunk = {chunk["chunk_id"]: chunk for chunk in chunks}

    @classmethod
    def from_chunks(cls, chunks):
        embeddings = embed_passages([chunk["text"] for chunk in chunks])
        return cls(chunks, embeddings)

    def search(self, query, k=4):
        query_vector = np.asarray(embed_queries([query])[0], dtype="float32").reshape(1, -1)
        scores = cosine_similarity(query_vector, self.embeddings)[0]
        top_indices = np.argsort(scores)[::-1][:k]

        results = []
        for idx in top_indices:
            chunk = self.chunks[idx]
            results.append(
                {
                    **chunk,
                    "score": float(scores[idx]),
                }
            )
        return results


In [11]:
base_strategy = "recursive_large"
base_chunks = chunk_sets[base_strategy]
dense_store = SimpleDenseVectorStore.from_chunks(base_chunks)

print(f"strategy={base_strategy}")
print(f"stored chunks={len(base_chunks)}")
print(f"embedding shape={dense_store.embeddings.shape}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

strategy=recursive_large
stored chunks=19
embedding shape=(19, 384)


In [12]:
def show_results(results, title=None, width=260):
    if title:
        print(f"\n{title}")
        print("-" * len(title))
    for rank, item in enumerate(results, start=1):
        preview = shorten(item["text"], width=width, placeholder=" ...")
        print(f"{rank}. score={item['score']:.4f} | page={item['page']} | {item['chunk_id']}")
        print(f"   {preview}")


In [13]:
dense_query = "How did MIT researchers implant a false memory in mice?"
dense_results = dense_store.search(dense_query, k=4)
show_results(dense_results, title=f"Dense Search | {dense_query}")



Dense Search | How did MIT researchers implant a false memory in mice?
----------------------------------------------------------------------
1. score=0.9035 | page=4 | p4-c1
   4 Nearly a decade ago, MIT researchers genetically altered mice so that when their neurons were active during learning, this activity turned on the ChR2 gene, which was tethered to a green fluorescent protein. By seeing which neurons fluoresced, ...
2. score=0.8629 | page=4 | p4-c2
   . Eventually, the mice associated the memory of the triangle box with the shocks even though they were shocked only while in the square box. “The animals were fearful of an environment that, technically speaking, never had anything ‘bad’ happen in it,” ...
3. score=0.8603 | page=4 | p4-c3
   . Summary → However, memories can be manipulated quite easily. Researchers could change the memories of mice by activating specific genes and neurons and then administering electric shocks to form a false, negative memory about their environm

위 결과에서 보통 page 4 근처 내용이 상위에 올라오면, dense retrieval이 문서 의미를 잘 잡고 있다고 볼 수 있습니다.


## 5. Sparse Search 추가하기

이번 노트북에서는 별도 BM25 패키지 대신 `TF-IDF + cosine similarity`로 sparse search를 간단히 구현합니다.

이 접근은 다음 장점이 있습니다.

- scikit-learn만으로 바로 실행 가능
- 단어 기반 검색의 감을 잡기 좋음
- dense search와 대비 학습하기 쉬움

주의할 점:

- TF-IDF는 의미 이해보다 단어 일치에 가깝습니다.
- 영어 문서에서는 영어 질의에서 강하고, 한국어 질의에서는 약할 수 있습니다.


In [14]:
class SimpleSparseSearcher:
    def __init__(self, chunks):
        self.chunks = chunks
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        self.matrix = self.vectorizer.fit_transform([chunk["text"] for chunk in chunks])

    def search(self, query, k=4):
        query_vector = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vector, self.matrix)[0]
        top_indices = np.argsort(scores)[::-1][:k]

        results = []
        for idx in top_indices:
            chunk = self.chunks[idx]
            results.append(
                {
                    **chunk,
                    "score": float(scores[idx]),
                }
            )
        return results


In [15]:
sparse_searcher = SimpleSparseSearcher(base_chunks)
sparse_results = sparse_searcher.search(dense_query, k=4)
show_results(sparse_results, title=f"Sparse Search | {dense_query}")



Sparse Search | How did MIT researchers implant a false memory in mice?
-----------------------------------------------------------------------
1. score=0.2640 | page=4 | p4-c1
   4 Nearly a decade ago, MIT researchers genetically altered mice so that when their neurons were active during learning, this activity turned on the ChR2 gene, which was tethered to a green fluorescent protein. By seeing which neurons fluoresced, ...
2. score=0.0763 | page=3 | p3-c2
   . In other experiments, researchers found that neural networks hold on to forgotten memories. Mice injected with a cocktail of protein inhibitors develop amnesia, likely forgetting information because their synapses wither away. But the researchers ...
3. score=0.0708 | page=4 | p4-c3
   . Summary → However, memories can be manipulated quite easily. Researchers could change the memories of mice by activating specific genes and neurons and then administering electric shocks to form a false, negative memory about their environmen

## 6. Hybrid Search 만들기

이제 dense와 sparse를 합칩니다. 이번 실습에서는 점수 스케일 차이를 덜 타는 `RRF(Reciprocal Rank Fusion)`를 사용합니다.

RRF의 장점:

- dense score와 sparse score의 단위가 달라도 비교적 안정적임
- 구현이 단순함
- 검색기마다 순위만 잘 나오면 꽤 잘 동작함


In [16]:
def reciprocal_rank_fusion(rankings, k=60):
    fused_scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            fused_scores[doc_id] = fused_scores.get(doc_id, 0.0) + 1 / (k + rank)
    return fused_scores

def hybrid_search(query, dense_store, sparse_searcher, final_k=4, fetch_k=8):
    dense_results = dense_store.search(query, k=fetch_k)
    sparse_results = sparse_searcher.search(query, k=fetch_k)

    dense_ranking = [item["chunk_id"] for item in dense_results]
    sparse_ranking = [item["chunk_id"] for item in sparse_results]
    fused_scores = reciprocal_rank_fusion([dense_ranking, sparse_ranking])

    dense_map = {item["chunk_id"]: item for item in dense_results}
    sparse_map = {item["chunk_id"]: item for item in sparse_results}
    candidate_ids = set(dense_ranking) | set(sparse_ranking)

    merged = []
    for chunk_id in candidate_ids:
        base_item = dense_store.id_to_chunk[chunk_id]
        merged.append(
            {
                **base_item,
                "score": float(fused_scores.get(chunk_id, 0.0)),
                "dense_score": float(dense_map.get(chunk_id, {}).get("score", 0.0)),
                "sparse_score": float(sparse_map.get(chunk_id, {}).get("score", 0.0)),
                "dense_rank": dense_ranking.index(chunk_id) + 1 if chunk_id in dense_ranking else None,
                "sparse_rank": sparse_ranking.index(chunk_id) + 1 if chunk_id in sparse_ranking else None,
            }
        )

    merged.sort(key=lambda item: item["score"], reverse=True)
    return merged[:final_k]


In [17]:
hybrid_results = hybrid_search(dense_query, dense_store, sparse_searcher, final_k=4, fetch_k=8)
show_results(hybrid_results, title=f"Hybrid Search (RRF) | {dense_query}")



Hybrid Search (RRF) | How did MIT researchers implant a false memory in mice?
-----------------------------------------------------------------------------
1. score=0.0328 | page=4 | p4-c1
   4 Nearly a decade ago, MIT researchers genetically altered mice so that when their neurons were active during learning, this activity turned on the ChR2 gene, which was tethered to a green fluorescent protein. By seeing which neurons fluoresced, ...
2. score=0.0318 | page=3 | p3-c2
   . In other experiments, researchers found that neural networks hold on to forgotten memories. Mice injected with a cocktail of protein inhibitors develop amnesia, likely forgetting information because their synapses wither away. But the researchers ...
3. score=0.0317 | page=4 | p4-c3
   . Summary → However, memories can be manipulated quite easily. Researchers could change the memories of mice by activating specific genes and neurons and then administering electric shocks to form a false, negative memory about thei

## 7. Dense vs Sparse vs Hybrid 비교

이번에는 문서 내용에 맞는 몇 가지 질의로 세 검색 방식을 비교합니다.

아래 질문은 MIT.pdf 실제 내용에 맞춰 잡았습니다.

- 기억의 물리적 표현은 무엇인가
- 과학자들은 어떤 도구로 기억을 관찰하는가
- MIT 연구진은 어떻게 거짓 기억을 심었는가


In [18]:
english_queries = [
    "What is a memory engram?",
    "What tools let scientists see memories?",
    "How did MIT researchers implant a false memory in mice?",
]

for query in english_queries:
    print(f"\n=== QUERY: {query} ===")
    print("dense top page:", dense_store.search(query, k=1)[0]["page"])
    print("sparse top page:", sparse_searcher.search(query, k=1)[0]["page"])
    print("hybrid top page:", hybrid_search(query, dense_store, sparse_searcher, final_k=1, fetch_k=8)[0]["page"])



=== QUERY: What is a memory engram? ===
dense top page: 1
sparse top page: 1
hybrid top page: 1

=== QUERY: What tools let scientists see memories? ===
dense top page: 2
sparse top page: 2
hybrid top page: 2

=== QUERY: How did MIT researchers implant a false memory in mice? ===
dense top page: 4
sparse top page: 4
hybrid top page: 4


## 8. 한국어 질의에서 dense의 장점 보기

문서는 영어인데 질문을 한국어로 던져 보면 dense search의 의미 매칭 능력이 더 잘 보일 수 있습니다.

반면 sparse search는 단어 일치 기반이기 때문에, 영어 문서에 한국어 질의를 넣으면 약할 가능성이 큽니다.


In [19]:
korean_query = "MIT 연구진은 어떻게 쥐에게 거짓 기억을 심었을까?"

dense_ko = dense_store.search(korean_query, k=3)
sparse_ko = sparse_searcher.search(korean_query, k=3)
hybrid_ko = hybrid_search(korean_query, dense_store, sparse_searcher, final_k=3, fetch_k=8)

show_results(dense_ko, title=f"Dense Search | {korean_query}")
show_results(sparse_ko, title=f"Sparse Search | {korean_query}")
show_results(hybrid_ko, title=f"Hybrid Search | {korean_query}")



Dense Search | MIT 연구진은 어떻게 쥐에게 거짓 기억을 심었을까?
--------------------------------------------
1. score=0.8221 | page=4 | p4-c1
   4 Nearly a decade ago, MIT researchers genetically altered mice so that when their neurons were active during learning, this activity turned on the ChR2 gene, which was tethered to a green fluorescent protein. By seeing which neurons fluoresced, ...
2. score=0.8184 | page=3 | p3-c2
   . In other experiments, researchers found that neural networks hold on to forgotten memories. Mice injected with a cocktail of protein inhibitors develop amnesia, likely forgetting information because their synapses wither away. But the researchers ...
3. score=0.8135 | page=5 | p5-c1
   5 Summary → While the technology is still in developing stages, researchers can now use neuroimaging and reconstruction algorithms to visually reconstruct the content of human memories. Further Study 교재와 강의를 통해 학습한 내용을 참고하여 주어진 한국어 문장을 영어로 옮겨 보세요. (아래 밑줄 친 ...

Sparse Search | MIT 연구진은 어떻게 쥐에게 거짓 

여기서 기대하는 관찰 포인트:

- dense는 한국어 질문과 영어 문서 사이의 의미 유사성을 어느 정도 잡을 수 있음
- sparse는 단어가 직접 맞지 않아서 약한 결과를 보일 수 있음
- hybrid는 sparse가 약하더라도 dense 결과를 유지할 수 있음


## 9. Chunking 전략별 retrieval 비교

이제 같은 dense embedding을 쓰되, chunking만 바꿔서 검색 결과를 비교합니다.

이번에는 간단한 `page hit@k` 스타일로 봅니다.

- 질의마다 기대 페이지를 하나 정하고
- top-k 안에 그 페이지가 들어오면 hit로 봅니다.

정교한 평가는 아니지만, chunking 차이를 체감하기에는 충분합니다.


In [20]:
evaluation_cases = [
    {
        "query": "Why is it difficult to locate a specific memory in the brain?",
        "expected_page": 1,
    },
    {
        "query": "What tools let scientists see memories?",
        "expected_page": 2,
    },
    {
        "query": "How did MIT researchers implant a false memory in mice?",
        "expected_page": 4,
    },
]

dense_stores_by_strategy = {
    strategy_name: SimpleDenseVectorStore.from_chunks(chunks)
    for strategy_name, chunks in chunk_sets.items()
}


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [21]:
def page_hit_at_k(results, expected_page, k=3):
    top_pages = [item["page"] for item in results[:k]]
    return int(expected_page in top_pages)

for strategy_name, store in dense_stores_by_strategy.items():
    hit1 = 0
    hit3 = 0
    for case in evaluation_cases:
        results = store.search(case["query"], k=3)
        hit1 += page_hit_at_k(results, case["expected_page"], k=1)
        hit3 += page_hit_at_k(results, case["expected_page"], k=3)
    total = len(evaluation_cases)
    print(
        f"{strategy_name:16} hit@1={hit1}/{total} ({hit1/total:.2f}) | hit@3={hit3}/{total} ({hit3/total:.2f})"
    )


recursive_small  hit@1=3/3 (1.00) | hit@3=3/3 (1.00)
recursive_large  hit@1=3/3 (1.00) | hit@3=3/3 (1.00)
token_medium     hit@1=3/3 (1.00) | hit@3=3/3 (1.00)


In [22]:
focus_query = "How did MIT researchers implant a false memory in mice?"

for strategy_name, store in dense_stores_by_strategy.items():
    print(f"\n=== strategy: {strategy_name} ===")
    focus_results = store.search(focus_query, k=3)
    show_results(focus_results)



=== strategy: recursive_small ===
1. score=0.9153 | page=4 | p4-c2
   . And they could reactivate specific memories by shining light o n the ChR2 genes associated with those neurons. With this ability, the MIT researchers inserted a false memory into mouse brains. First they placed the mice in a triangular box, which ...
2. score=0.8651 | page=4 | p4-c1
   4 Nearly a decade ago, MIT researchers genetically altered mice so that when their neurons were active during learning, this activity turned on the ChR2 gene, which was tethered to a green fluorescent protein. By seeing which neurons fluoresced, ...
3. score=0.8595 | page=4 | p4-c3
   . Then they put the mice in a square box and administered shocks to their feet while shining a light on the ChR2 neurons associated with the first environment. Eventually, the mice associated the memory of the triangle box with the shocks even though they ...

=== strategy: recursive_large ===
1. score=0.9035 | page=4 | p4-c1
   4 Nearly a decade ago, 

## 10. 직접 바꿔 볼 것

아래 항목은 꼭 직접 수정해 보세요.

1. `chunk_size`를 더 작게 또는 크게 바꿔 보기
2. `chunk_overlap`을 0으로 바꿔 보기
3. 한국어 질의를 더 많이 넣어 보기
4. `TfidfVectorizer`의 `ngram_range`를 바꿔 보기
5. `hybrid_search`에서 RRF 대신 가중합 방식으로 바꿔 보기

이 실험을 해 보면 retrieval는 한 가지 정답이 있는 문제가 아니라, **문서 특성 + 질의 특성 + 검색 전략의 조합 문제**라는 감이 잡힙니다.


## 실습 정리

이번 실습에서 확인해야 할 핵심은 아래입니다.

- 임베딩 모델만 고르는 것으로 RAG 검색 품질이 끝나지 않는다.
- 같은 문서도 chunking 방식에 따라 retrieval 결과가 달라진다.
- dense search는 의미 기반 검색에 강하다.
- sparse search는 단어 기반 검색에 강하다.
- hybrid search는 둘의 장점을 합쳐 실무에서 자주 쓰인다.

이 노트북을 한 번 실행하고 끝내지 말고, 숫자와 파라미터를 직접 바꾸면서 결과를 비교해 보는 것이 가장 중요합니다.
